#### MPAS time series plotting 
 Subsetting an Unstructured Grid: Analysis Over Hawaii
- Bounding Box: bugging
- Bounding Circle
- Nearest Neighbor
- Retrieving Orignal Grid Indices
  
Ming Ge Feb 2025

Reference:
- https://uxarray.readthedocs.io/en/latest/user-guide/subset.html
- https://uxarray.readthedocs.io/en/v2024.01.1/examples/009-subsetting.html 
- https://uxarray.readthedocs.io/en/latest/user-guide/data-structures.html
- https://projectpythia.org/unstructured-grid-viz-cookbook/notebooks/02-intro-to-uxarray/selection.html
- https://projectpythia.org/unstructured-grid-viz-cookbook/notebooks/04-recipes/mpas-atmo.html

In [1]:
%matplotlib inline
import uxarray as ux
import geoviews.feature as gf
import cartopy.crs as ccrs
#import holoviews as hv
import warnings

plot_opts = {"width" : 600, "height": 300}

#hv.extension('bokeh')

warnings.filterwarnings("ignore")

#n.warnings = warnings

#### MPAS visulization
https://projectpythia.org/unstructured-grid-viz-cookbook/notebooks/04-recipes/mpas-atmo.html

In [2]:
dir_ghg = '/glade/campaign/mmm/c3we/mingge/MPAS-PGW/OLD/1993_PGW/'
dir_pgw = '/glade/campaign/mmm/c3we/mingge/MPAS-PGW/OLD/1993_PGW/'

flnm_grid = '/glade/campaign/mmm/c3we/mingge/MPAS-PGW/25-90km_PGW_tmp/x4.163842.grid_hawaii.nc'
flnm_data = dir_pgw + 'diag.1993-12-01_00.00.00.nc'

# Open dataset and grab a data variable of interest
uxds = ux.open_dataset(flnm_grid, flnm_data, use_dual = False)
uxda = uxds["relhum_200hPa"][0]


In [3]:
clim = (uxds['relhum_200hPa'][0].values.min(), uxds['relhum_200hPa'][0].values.max())
features = gf.coastline(projection=ccrs.PlateCarree(), line_width=1, scale='50m') * \
           gf.states(projection=ccrs.PlateCarree(), line_width=1, scale='50m')

In [4]:
uxds['relhum_200hPa'][0].plot.rasterize(method='polygon',
                                        exclude_antimeridian=True,
                                        title = "Global Grid",
                                        **plot_opts) * features


:Overlay
   .Image.I     :Image   [x,y]   (x_y relhum_200hPa)
   .Coastline.I :Feature   [Longitude,Latitude]
   .States.I    :Feature   [Longitude,Latitude]

#### analysis operations.

In [12]:
uxds['relhum_200hPa'][0].values.mean()

np.float32(39.945675)

#### Regional Subsets

In [13]:
uxds['relhum_200hPa'][0].subset

<uxarray.UxDataArray.subset>
Supported Methods:
  * nearest_neighbor(center_coord, k, element, inverse_indices, **kwargs)
  * bounding_circle(center_coord, r, element, inverse_indices, **kwargs)
  * bounding_box(lon_bounds, lat_bounds, inverse_indices, **kwargs)

In [14]:
uxds['relhum_200hPa'].uxgrid.subset

<uxarray.Grid.subset>
Supported Methods:
  * nearest_neighbor(center_coord, k, element, inverse_indices, **kwargs)
  * bounding_circle(center_coord, r, element, inverse_indices, **kwargs)
  * bounding_box(lon_bounds, lat_bounds, inverse_indices)

#### Bounding Box

In [15]:
#subset.bounding_box(lon_bounds = (-10, 10), lat_bounds=(-5, 5))

In [16]:
uxgrid = uxds.uxgrid
uxgrid.subset.bounding_box(lon_bounds = (-10, 10), lat_bounds=(-5, 5)).plot.nodes(title="Node Plot")

:Points   [lon,lat]

In [17]:
uxgrid.subset

<uxarray.Grid.subset>
Supported Methods:
  * nearest_neighbor(center_coord, k, element, inverse_indices, **kwargs)
  * bounding_circle(center_coord, r, element, inverse_indices, **kwargs)
  * bounding_box(lon_bounds, lat_bounds, inverse_indices)

In [18]:
uxds['relhum_200hPa'][0].subset

<uxarray.UxDataArray.subset>
Supported Methods:
  * nearest_neighbor(center_coord, k, element, inverse_indices, **kwargs)
  * bounding_circle(center_coord, r, element, inverse_indices, **kwargs)
  * bounding_box(lon_bounds, lat_bounds, inverse_indices, **kwargs)

#### Bounding Box
Previously failed on an older uxarray version; works as of uxarray 2025.4.0

In [19]:
# https://uxarray.readthedocs.io/en/latest/user-guide/subset.html
lon_bounds = (-180 - 2, -120 + 2)
lat_bounds = (0 - 2, 40 + 2)
lon_bounds = (-87.6298 - 2, -87.6298 + 2)
lat_bounds = (41.8781 - 2, 41.8781 + 2)
inverse_indices=(["face", "node", "edge"], True)

bbox_subset_nodes = uxds['relhum_200hPa'][0].subset.bounding_box(lon_bounds, lat_bounds, inverse_indices=(["face", "node", "edge"], True))

bbox_subset_nodes.plot.rasterize(method='polygon',
                                 exclude_antimeridian=True,
                                 clim=clim,
                                 title="Bounding Box Subset",
                                 **plot_opts) * features

:Overlay
   .Image.I     :Image   [x,y]   (x_y relhum_200hPa)
   .Coastline.I :Feature   [Longitude,Latitude]
   .States.I    :Feature   [Longitude,Latitude]

#### Bounding Circle
Using a center coordinate (lon, lat) and a radius (in degrees). The resulting subset will contain all elements within the radius of that circle.

In [20]:
center_coord = [-156, 20]

r = 4
bcircle_subset = uxds["relhum_200hPa"][0].subset.bounding_circle(center_coord, r)

bcircle_subset.plot(
    rasterize=True,
    periodic_elements="exclude",
    clim=clim,
    title="Bounding Circle Subset",
    **plot_opts,
) * features


:Overlay
   .Image.I     :Image   [x,y]   (x_y relhum_200hPa)
   .Coastline.I :Feature   [Longitude,Latitude]
   .States.I    :Feature   [Longitude,Latitude]

#### Nearest Neighbor
perform a nearest neighbor subset at some center coordinate (lon, lat) and query for some number of elements k

In [21]:
center_coord = [-156, 20]
nn_subset = uxds["relhum_200hPa"][0].subset.nearest_neighbor(
    center_coord, k=10, element="nodes"  # query against mesh nodes (corners), not cell/face centers
)

nn_subset.plot(
    rasterize=True,
    periodic_elements="exclude",
    clim=clim,
    title="Nearest Neighbor Subset",
    **plot_opts,
) * features

:Overlay
   .Image.I     :Image   [x,y]   (x_y relhum_200hPa)
   .Coastline.I :Feature   [Longitude,Latitude]
   .States.I    :Feature   [Longitude,Latitude]

In [22]:
nn_subset_120 = uxds["relhum_200hPa"][0].subset.nearest_neighbor(
    center_coord, k=120, element="face centers"  # query against cell/face centers instead of nodes
)

nn_subset_120.plot(
    rasterize=True,
    periodic_elements="exclude",
    clim=clim,
    title="Nearest Neighbor Subset (120 Faces)",
    **plot_opts,
) * features

:Overlay
   .Image.I     :Image   [x,y]   (x_y relhum_200hPa)
   .Coastline.I :Feature   [Longitude,Latitude]
   .States.I    :Feature   [Longitude,Latitude]

In [23]:
# When we set k=1, it selects the closest neighbor.

nn_subset_1 = uxds["relhum_200hPa"][0].subset.nearest_neighbor(
    center_coord, k=1, element="face centers"
)

nn_subset_1.plot(
    rasterize=True,
    periodic_elements="exclude",
    clim=clim,
    title="Nearest Neighbor Subset (Closest Face)",
    **plot_opts,
) * features

:Overlay
   .Image.I     :Image   [x,y]   (x_y relhum_200hPa)
   .Coastline.I :Feature   [Longitude,Latitude]
   .States.I    :Feature   [Longitude,Latitude]

#### Analysis Operators on Regional Subsets

In [24]:
print("Global Mean:            ", uxds["relhum_200hPa"][0].values.mean())
#print("Bounding Box Mean:      ", bbox_subset_nodes.values.mean())
print("Bounding Circle Mean:   ", bcircle_subset.values.mean())

Global Mean:             39.945675
Bounding Circle Mean:    31.681105


#### Retrieving Orignal Grid Indices
 These indices can be stored within the subset with the inverse_indices variable. 
- Passing in True, which will store the face center indices

- Passing in a list of which indices to store, along with True, to indicate what kind of original grid indices to store.

In [25]:
subset_indices = uxds["relhum_200hPa"][0].subset.bounding_circle(
    center_coord,
    r,
    element="face centers",
    inverse_indices=(["face", "node", "edge"], True),  # keep mapping back to original (unsubset) grid indices
)
subset_indices.uxgrid.inverse_indices 

<xarray.Dataset> Size: 72kB
Dimensions:  (face: 1448, node: 3033, edge: 4480)
Coordinates:
  * face     (face) int64 12kB 46586 119274 15866 ... 119416 119410 119408
  * node     (node) int64 24kB 6 7 8 9 10 ... 326707 326708 327365 327366 327606
  * edge     (edge) int64 36kB 278469 278609 278610 ... 332891 333020 333021
Data variables:
    *empty*

In [26]:
subset_indices.uxgrid.inverse_indices.face 

<xarray.DataArray 'face' (face: 1448)> Size: 12kB
array([ 46586, 119274,  15866, ..., 119416, 119410, 119408], shape=(1448,))
Coordinates:
  * face     (face) int64 12kB 46586 119274 15866 ... 119416 119410 119408

In [27]:
subset_indices.uxgrid.inverse_indices.node

<xarray.DataArray 'node' (node: 3033)> Size: 24kB
array([     6,      7,      8, ..., 327365, 327366, 327606], shape=(3033,))
Coordinates:
  * node     (node) int64 24kB 6 7 8 9 10 ... 326707 326708 327365 327366 327606

In [28]:
subset_indices.uxgrid.inverse_indices.edge

<xarray.DataArray 'edge' (edge: 4480)> Size: 36kB
array([278469, 278609, 278610, ..., 332891, 333020, 333021], shape=(4480,))
Coordinates:
  * edge     (edge) int64 36kB 278469 278609 278610 ... 332891 333020 333021